# 🔥 Chapter 9 — Advanced Techniques & Common Pitfalls

---

### 🎯 What you'll learn
- Learning rate schedulers
- Mixed precision training (AMP)
- Gradient accumulation for large effective batch sizes
- Freezing layers for transfer learning
- The complete pitfall checklist

## 9.1 — Learning Rate Schedulers

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Step LR: decay by factor every N epochs
scheduler1 = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# Cosine Annealing: smooth decay
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# ReduceOnPlateau: reduce when metric stops improving
scheduler3 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

# ⚠️ Corner Case: scheduler.step() placement
# For most schedulers: call AFTER optimizer.step()
# For ReduceLROnPlateau: pass the metric!
# scheduler3.step(val_loss)  ← needs the monitored metric

# Visualize LR schedule
lrs = []
optimizer_viz = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler_viz = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_viz, T_max=50)
for epoch in range(50):
    lrs.append(optimizer_viz.param_groups[0]['lr'])
    optimizer_viz.step()
    scheduler_viz.step()
print(f"LR at epoch 0: {lrs[0]:.6f}")
print(f"LR at epoch 25: {lrs[25]:.6f}")
print(f"LR at epoch 49: {lrs[49]:.6f}")

## 9.2 — Mixed Precision Training (AMP)
Use `float16` where possible for ~2x speedup on compatible GPUs.

In [ ]:
from torch.cuda.amp import autocast, GradScaler

# ⚠️ AMP is most beneficial on NVIDIA GPUs with Tensor Cores (V100, A100, etc.)
# On CPU or MPS, it provides little to no benefit.

model = nn.Sequential(nn.Linear(100, 256), nn.ReLU(), nn.Linear(256, 10))
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()  # Scales loss to prevent float16 underflow

x = torch.randn(32, 100)
y = torch.randint(0, 10, (32,))

# ── AMP Training Step ──
# with autocast():  # Use autocast for forward pass
#     output = model(x)
#     loss = criterion(output, y)
# 
# scaler.scale(loss).backward()    # Scale loss before backward
# scaler.step(optimizer)           # Unscale and step
# scaler.update()                  # Update scaler for next iteration
# optimizer.zero_grad()

# ⚠️ Corner Case: Not all operations support float16
# Convolutions, matmuls → float16 (fast)
# Reductions (sum, softmax, loss) → float32 (accurate)
# autocast handles this automatically!
print("AMP pattern ready — use on GPU for speedup!")

## 9.3 — Gradient Accumulation
Simulate larger batch sizes when GPU memory is limited.

In [ ]:
model = nn.Linear(10, 1)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

accumulation_steps = 4  # Effective batch = batch_size × accumulation_steps

for i in range(20):
    x = torch.randn(8, 10)  # Small batch (8)
    y = torch.randn(8, 1)
    
    loss = criterion(model(x), y)
    loss = loss / accumulation_steps  # ⚠️ Scale loss by accumulation steps!
    loss.backward()                    # Gradients accumulate
    
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()       # Update weights every N mini-batches
        optimizer.zero_grad()  # Reset gradients
        print(f"Step {i+1}: weight updated (effective batch = {8 * accumulation_steps})")

print("\nGradient accumulation simulates batch_size=32 using batch_size=8!")

## 9.4 — Freezing Layers (Transfer Learning)

In [ ]:
model = nn.Sequential(
    nn.Linear(100, 64),   # Feature extractor
    nn.ReLU(),
    nn.Linear(64, 32),    # Feature extractor
    nn.ReLU(),
    nn.Linear(32, 10),    # Classifier head
)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the last layer (classifier head)
for param in model[4].parameters():
    param.requires_grad = True

# ✅ Only pass trainable params to optimizer!
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

# Verify
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable} / {total} parameters")

## 9.5 — 🔥 The Complete Pitfall Checklist

| # | Pitfall | Fix |
|---|---------|-----|
| 1 | Forgot `model.train()` / `model.eval()` | Always toggle before train/eval loops |
| 2 | Forgot `optimizer.zero_grad()` | Gradients accumulate silently → wrong updates |
| 3 | Forgot `torch.no_grad()` in validation | Wastes GPU memory, may OOM |
| 4 | Used `model.forward(x)` instead of `model(x)` | Skips hooks → silent bugs |
| 5 | Wrong `dim=` in softmax/loss | Softmax over wrong axis → garbage probabilities |
| 6 | CrossEntropyLoss with one-hot targets | Expects class indices, not one-hot |
| 7 | Applied softmax before CrossEntropyLoss | CE includes softmax → double softmax! |
| 8 | Created optimizer before `model.to(device)` | Optimizer tracks wrong device tensors |
| 9 | In-place ops on tensors with `requires_grad` | Breaks autograd computation graph |
| 10 | Not using `drop_last=True` with BatchNorm | Single-sample batches crash BatchNorm |
| 11 | float16 overflow | Values > 65504 become `inf` |
| 12 | Loading model without `map_location` | Crash when GPU→CPU or vice versa |
| 13 | Views share memory with original tensor | Modifying view modifies original |
| 14 | Non-contiguous tensor with `.view()` | Use `.reshape()` or `.contiguous().view()` |
| 15 | Not scaling loss in gradient accumulation | Gradients are N× too large |

## ✏️ Exercises

In [ ]:
# Exercise 1: Implement a training loop with:
# - Cosine annealing LR scheduler
# - Gradient accumulation (4 steps)
# - Gradient clipping (max_norm=1.0)
# YOUR CODE HERE


# Exercise 2: Take a pre-trained model, freeze all layers,
# replace the last layer, and fine-tune only the new layer.
# YOUR CODE HERE
